# 01 — Preparação do Ambiente Experimental

Este notebook documenta a preparação do ambiente utilizado no experimento de avaliação de defesas contra prompt injection em LLMs.

O objetivo desta etapa é registrar:

- estrutura do projeto;
- ambiente Python;
- dependências utilizadas;
- disponibilidade de GPU e CUDA;
- versões das bibliotecas;
- diretórios utilizados para dados, scripts, adaptadores, logs, resultados brutos e métricas.

Este notebook faz parte dos artefatos de reprodução do experimento.

## 1. Ambiente de execução: RunPod

O experimento foi executado em um ambiente de GPU em nuvem utilizando o RunPod. A escolha do RunPod foi motivada pela necessidade de acessar uma GPU dedicada para treinamento e avaliação de adaptadores LoRA em um modelo Llama 3/3.1 8B Instruct, sem depender de recursos locais.

O uso de uma GPU em nuvem foi importante porque os cenários C2, C3 e C4 envolvem treinamento ou ajuste do modelo, o que exige maior capacidade computacional do que a disponível em uma máquina comum. Além disso, o RunPod permite selecionar diferentes tipos de GPU, configurar disco, volume persistente, portas expostas e acessar o ambiente por terminal, JupyterLab ou outras formas de conexão remota.

De forma geral, o processo de criação do Pod seguiu as seguintes etapas:

1. Acessar a plataforma RunPod.
2. Entrar na área de Pods.
3. Selecionar a opção de Deploy ou criação de novo Pod.
4. Escolher uma GPU compatível com o experimento.
5. Escolher um template com suporte a PyTorch/CUDA.
6. Configurar o volume e o disco do container.
7. Expor portas necessárias, como JupyterLab ou SSH.
8. Iniciar o Pod.
9. Acessar o ambiente por terminal, SSH ou JupyterLab.

O Pod utilizado nesta execução foi configurado com uma GPU dedicada. As informações exatas da GPU são registradas automaticamente mais adiante por meio do comando `nvidia-smi`. No entanto, as demais informações utilizadas estão na tabela a seguir.

| Pod Template        | GPU Count | Container disk | Volume disk |
|---------------------|-----------|----------------|-------------|
| Runpod Pytorch 2.8. | 1         | 50 GB          | 100GB       |

Além disso, a configuração permitiu acesso ao terminal SSH e Jupyter Notebook.

---

Neste experimento, o diretório principal foi organizado em:

```bash
/workspace/pi-defense-exp
```

A pasta `/workspace` foi utilizada porque, em ambientes RunPod, ela é normalmente associada ao armazenamento persistente do Pod. Isso reduz o risco de perda de dados ao reiniciar o ambiente, desde que os arquivos importantes sejam mantidos dentro desse diretório.

Durante o uso do RunPod, também é importante observar alguns cuidados práticos:

* manter dados, scripts, resultados e adaptadores dentro de `/workspace`;
* salvar snapshots ou backups após etapas importantes;
* registrar versões das bibliotecas e informações da GPU;
* usar logs para documentar treinamentos e avaliações;
* evitar armazenar artefatos importantes apenas em diretórios temporários do container;
* baixar os resultados finais para a máquina local ao final do experimento.

## 2. Bootstrap do ambiente e seleção do kernel

Antes de executar qualquer célula Python deste notebook, é necessário garantir que o ambiente virtual do projeto foi criado, que as dependências foram instaladas e que o kernel correto foi registrado no Jupyter.

Essa etapa é necessária porque o notebook deve ser executado usando o kernel:

```text
Python (pi-defense-exp)
```

Se o notebook for executado com outro kernel, células que importam bibliotecas como `torch`, `transformers`, `datasets`, `peft` e `trl` podem falhar, mesmo que essas bibliotecas tenham sido instaladas dentro da `.venv`.

Portanto, o fluxo correto é:

```text
1. Criar a pasta do projeto.
2. Criar a .venv.
3. Instalar as dependências.
4. Registrar o kernel Jupyter da .venv.
5. Selecionar manualmente o kernel Python (pi-defense-exp).
6. Só então executar as células Python de validação do ambiente.
```

Após executar os comandos de bootstrap, reinicie ou troque o kernel do notebook para `Python (pi-defense-exp)` antes de continuar.

### Comandos de bootstrap

Execute estes comandos no terminal do RunPod antes de rodar as células Python do notebook:

```bash
cd /workspace
mkdir -p pi-defense-exp
cd pi-defense-exp

python3 -m venv .venv
source .venv/bin/activate

python -m pip install --upgrade pip setuptools wheel

pip install \
  torch \
  transformers \
  accelerate \
  datasets \
  peft \
  trl \
  safetensors \
  sentencepiece \
  protobuf \
  pandas \
  numpy \
  scikit-learn \
  scipy \
  statsmodels \
  matplotlib \
  tqdm \
  rich \
  evaluate \
  huggingface_hub \
  jupyter \
  jupyterlab \
  ipykernel

python -m ipykernel install \
  --user \
  --name pi-defense-exp \
  --display-name "Python (pi-defense-exp)"
```

Depois disso, no Jupyter, selecione:

```text
Kernel → Change Kernel → Python (pi-defense-exp)
```


In [1]:
import sys
from pathlib import Path

python_executable = Path(sys.executable)

print("Python executable:", python_executable)

expected_fragment = "/workspace/pi-defense-exp/.venv/bin/python"

if expected_fragment not in str(python_executable):
    raise RuntimeError(
        "Kernel incorreto. Selecione o kernel 'Python (pi-defense-exp)' antes de continuar."
    )

print("Kernel correto: Python (pi-defense-exp)")

Python executable: /workspace/pi-defense-exp/.venv/bin/python
Kernel correto: Python (pi-defense-exp)


In [2]:
from pathlib import Path
import os
import platform
import subprocess
from datetime import datetime

import pandas as pd
import numpy as np

print("Bibliotecas básicas carregadas.")

Bibliotecas básicas carregadas.


In [3]:
PROJECT_DIR = Path("/workspace/pi-defense-exp")

print("PROJECT_DIR:", PROJECT_DIR)
print("Existe?", PROJECT_DIR.exists())

if PROJECT_DIR.exists():
    print("Conteúdo atual:")
    for item in sorted(PROJECT_DIR.iterdir()):
        print("-", item.name)

PROJECT_DIR: /workspace/pi-defense-exp
Existe? True
Conteúdo atual:
- .ipynb_checkpoints
- .venv
- cache
- data
- environment
- exports
- notebooks
- runs
- scripts


## 3. Estrutura de pastas do projeto

A estrutura do projeto foi organizada para separar claramente os diferentes tipos de artefatos gerados durante o experimento. Como o estudo envolve múltiplas execuções com seeds diferentes, a organização considera não apenas o nome dos arquivos, mas também o nome das pastas onde cada artefato é armazenado.

A estrutura geral do projeto é:

```bash
/workspace/pi-defense-exp
├── data
├── scripts
├── notebooks
├── cache
├── exports
└── runs
```

A pasta `runs` concentra as execuções experimentais versionadas. Cada execução é organizada por nome do experimento, seed usada para construção do dataset e seed usada no treinamento.

A estrutura recomendada é:

```bash
runs/
└── opi_sst2_sms_5k_872
    └── dataset_seed42
        ├── data
        ├── baselines
        │   └── c0_c1
        ├── train_seed42
        ├── train_seed123
        └── train_seed2026
```

Essa organização permite diferenciar claramente dois tipos de seed:

* `dataset_seed`: controla a construção do conjunto de dados, o pareamento entre exemplos e a separação dos ataques vistos e não vistos;
* `train_seed`: controla a aleatoriedade do treinamento, como inicialização dos adaptadores LoRA, ordem dos batches e operações estocásticas do treinamento.

No experimento, o dataset foi mantido fixo com `dataset_seed42`, enquanto os cenários treinados podem ser executados com diferentes seeds de treinamento, como `train_seed42`, `train_seed123` e `train_seed2026`.

### `data/raw`

Armazena dados brutos obtidos de fontes externas ou benchmarks antes de qualquer transformação. Essa pasta pode ser usada para guardar dados originais ou derivados de benchmarks como Open-Prompt-Injection antes da conversão para o formato canônico.

### `data/processed`

Armazena os dados processados mais recentes, prontos para uso pelos scripts. Essa pasta funciona como área de trabalho ativa.

Exemplos de arquivos:

```bash
canonical_train.jsonl
canonical_test.jsonl
train_struq_sft.jsonl
train_secalign_dpo.jsonl
train_ih_sft.jsonl
test_all_scenarios.jsonl
```

Como os arquivos em `data/processed` podem ser sobrescritos quando o dataset é recriado, uma cópia versionada deve ser salva dentro da pasta `runs`.

### `runs/<experiment_name>/dataset_seed<seed>/data`

Armazena a cópia versionada dos dados utilizados em uma execução específica.

Exemplo:

```bash
runs/opi_sst2_sms_5k_872/dataset_seed42/data/
```

Essa pasta guarda exatamente os dados usados no experimento com 5.000 exemplos de treino, 872 exemplos de teste e seed de dataset igual a 42.

### `scripts`

Contém os scripts Python utilizados para preparar dados, treinar adaptadores, avaliar modelos e consolidar resultados.

Exemplos:

```bash
build_canonical_from_opi_sst2_sms.py
build_training_views.py
train_sft_lora_fp16.py
train_dpo_lora_fp16.py
evaluate_c0_c1.py
evaluate_adapter_scenario.py
```

### `runs/<experiment_name>/dataset_seed<seed>/baselines/c0_c1`

Armazena os artefatos dos cenários que não envolvem treinamento.

Essa pasta contém os resultados de:

* C0: modelo base sem defesa;
* C1: StruQ format-only;
* avaliação limpa do modelo base.

Como C0 e C1 não envolvem treinamento, eles não ficam dentro de uma pasta `train_seed`.

Estrutura esperada:

```bash
baselines/c0_c1/
├── logs
├── raw
└── metrics
```

### `runs/<experiment_name>/dataset_seed<seed>/train_seed<seed>`

Armazena os artefatos dos cenários treinados para uma seed específica.

Exemplo:

```bash
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed123/
```

Essa pasta contém tudo que foi produzido usando o mesmo dataset, mas com seed de treinamento igual a 123.

Estrutura esperada:

```bash
train_seed123/
├── adapters
├── logs
├── raw
└── metrics
```

### `adapters`

Dentro de cada `train_seed`, armazena os adaptadores LoRA treinados para cada cenário.

Exemplo:

```bash
train_seed123/adapters/
├── c2_struq_sft
├── c3_secalign_dpo
└── c4_instruction_hierarchy_sft
```

Essa organização evita confusão entre adaptadores treinados com seeds diferentes.

### `raw`

Armazena os resultados brutos da avaliação, normalmente em formato JSONL. Cada linha representa uma amostra avaliada e pode conter:

* identificador da amostra;
* cenário experimental;
* prompt enviado ao modelo;
* resposta esperada;
* resposta associada à injeção;
* saída gerada pelo modelo;
* label extraída;
* tipo de ataque;
* indicação de ataque visto ou não visto;
* seed associada à execução.

Exemplo:

```bash
train_seed123/raw/c2_struq_sft_raw_generations.jsonl
```

### `metrics`

Armazena as métricas agregadas calculadas a partir dos resultados brutos.

Exemplos:

```bash
train_seed123/metrics/c2_struq_sft_metrics.csv
train_seed123/metrics/c3_secalign_dpo_metrics.csv
train_seed123/metrics/c4_instruction_hierarchy_sft_metrics.csv
```

Essa pasta concentra métricas como Attack Success Value, Robust Accuracy, Injection Following Rate, Task Success Rate e Utility Drop.

### `logs`

Armazena os logs completos dos comandos executados no terminal, especialmente treinamentos e avaliações. Os logs foram salvos usando `tee`, permitindo visualizar a execução no terminal e registrar o conteúdo em arquivo ao mesmo tempo.

Exemplo:

```bash
train_seed123/logs/03_train_struq_sft.log
```

### `notebooks`

Armazena os notebooks Jupyter usados para documentar, inspecionar e analisar o experimento.

Exemplo:

```bash
notebooks/01_environment_setup.ipynb
```

### `exports`

Armazena pacotes compactados ou snapshots do experimento para transferência para outra máquina.

Exemplo:

```bash
exports/opi_sst2_sms_5k_872_dataset_seed42_snapshot.tar.gz
```

### `cache`

Armazena caches do Hugging Face, Transformers e Datasets dentro do diretório do projeto. Isso evita espalhar arquivos grandes em outras partes do sistema e facilita o controle do armazenamento usado pelo experimento.

In [4]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/workspace/pi-defense-exp")

EXPERIMENT_NAME = "opi_sst2_sms_5k_872"
DATASET_SEED = 42
TRAINING_SEEDS = [42, 123, 2026]

RUN_ROOT = PROJECT_DIR / "runs" / EXPERIMENT_NAME / f"dataset_seed{DATASET_SEED}"

directories = [
    PROJECT_DIR / "data" / "raw",
    PROJECT_DIR / "data" / "processed",
    PROJECT_DIR / "data" / "runs",
    PROJECT_DIR / "scripts",
    PROJECT_DIR / "notebooks",
    PROJECT_DIR / "environment",
    PROJECT_DIR / "cache",
    PROJECT_DIR / "exports",
    RUN_ROOT / "data",
    RUN_ROOT / "baselines" / "c0_c1" / "logs",
    RUN_ROOT / "baselines" / "c0_c1" / "raw",
    RUN_ROOT / "baselines" / "c0_c1" / "metrics",
]

for seed in TRAINING_SEEDS:
    seed_root = RUN_ROOT / f"train_seed{seed}"

    directories.extend([
        seed_root / "adapters" / "c2_struq_sft",
        seed_root / "adapters" / "c3_secalign_dpo",
        seed_root / "adapters" / "c4_instruction_hierarchy_sft",
        seed_root / "logs",
        seed_root / "raw",
        seed_root / "metrics",
    ])

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

directory_check = pd.DataFrame([
    {
        "directory": str(directory.relative_to(PROJECT_DIR)),
        "exists": directory.exists(),
        "path": str(directory),
    }
    for directory in directories
])

directory_check

,directory,exists,path
0,data/raw,True,/workspace/pi-defense-exp/data/raw
1,data/processed,True,/workspace/pi-defense-exp/data/processed
2,data/runs,True,/workspace/pi-defense-exp/data/runs
3,scripts,True,/workspace/pi-defense-exp/scripts
4,notebooks,True,/workspace/pi-defense-exp/notebooks
5,environment,True,/workspace/pi-defense-exp/environment
6,cache,True,/workspace/pi-defense-exp/cache
7,exports,True,/workspace/pi-defense-exp/exports
8,runs/opi_sst2_sms_5k_872/dataset_seed42/data,True,/workspace/pi-defense-exp/runs/opi_sst2_sms_5k...
9,runs/opi_sst2_sms_5k_872/dataset_seed42/baseli...,True,/workspace/pi-defense-exp/runs/opi_sst2_sms_5k...


## 4. Ambiente virtual Python

O ambiente Python foi isolado usando `.venv`, para evitar conflito com bibliotecas globais do sistema.

Comandos utilizados no terminal:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip setuptools wheel
```

Depois de ativado, o Python esperado é:

```bash
/workspace/pi-defense-exp/.venv/bin/python
```

In [5]:
import sys

environment_info = {
    "python_executable": sys.executable,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "working_directory": os.getcwd(),
}

pd.DataFrame([environment_info])

,python_executable,python_version,platform,working_directory
0,/workspace/pi-defense-exp/.venv/bin/python,3.12.3,Linux-6.8.0-107-generic-x86_64-with-glibc2.39,/workspace/pi-defense-exp/notebooks


## 5. Dependências principais

As bibliotecas principais utilizadas no experimento foram:

- `torch`;
- `transformers`;
- `accelerate`;
- `datasets`;
- `peft`;
- `trl`;
- `safetensors`;
- `sentencepiece`;
- `protobuf`;
- `pandas`;
- `numpy`;
- `scikit-learn`;
- `scipy`;
- `statsmodels`;
- `matplotlib`;
- `tqdm`;
- `rich`;
- `evaluate`;
- `huggingface_hub`;
- `jupyter`;
- `jupyterlab`;
- `ipykernel`.

Comando utilizado:

```bash
pip install \
  torch \
  transformers \
  accelerate \
  datasets \
  peft \
  trl \
  safetensors \
  sentencepiece \
  protobuf \
  pandas \
  numpy \
  scikit-learn \
  scipy \
  statsmodels \
  matplotlib \
  tqdm \
  rich \
  evaluate \
  huggingface_hub \
  jupyter \
  jupyterlab \
  ipykernel
```

In [6]:
packages_to_check = [
    "torch",
    "transformers",
    "accelerate",
    "datasets",
    "peft",
    "trl",
    "pandas",
    "numpy",
    "sklearn",
    "scipy",
    "statsmodels",
    "matplotlib",
    "tqdm",
    "evaluate",
    "huggingface_hub",
]

import importlib

package_rows = []

for package in packages_to_check:
    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "version_not_available")
        status = "ok"
    except Exception as e:
        version = None
        status = f"error: {e}"

    package_rows.append({
        "package": package,
        "version": version,
        "status": status,
    })

packages_df = pd.DataFrame(package_rows)
packages_df

,package,version,status
0,torch,2.12.1+cu130,ok
1,transformers,5.12.1,ok
2,accelerate,1.14.0,ok
3,datasets,5.0.0,ok
4,peft,0.19.1,ok
5,trl,1.6.0,ok
6,pandas,3.0.3,ok
7,numpy,2.4.6,ok
8,sklearn,1.9.0,ok
9,scipy,1.18.0,ok


## 6. Configuração de cache

O cache do Hugging Face foi direcionado para dentro do projeto, para manter os arquivos organizados e facilitar reprodutibilidade.
As variáveis definidas via `os.environ` valem para o processo atual do notebook. Para uso no terminal, os mesmos valores devem ser exportados no shell ou adicionados ao `~/.bashrc`.

Comandos utilizados:

```bash
export HF_HOME=/workspace/pi-defense-exp/cache/huggingface
export TRANSFORMERS_CACHE=/workspace/pi-defense-exp/cache/huggingface
export HF_DATASETS_CACHE=/workspace/pi-defense-exp/cache/huggingface/datasets
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
export MODEL_ID=meta-llama/Meta-Llama-3.1-8B-Instruct
```

Essas variáveis também foram adicionadas ao ~/.bashrc:

```bash
echo 'export HF_HOME=/workspace/pi-defense-exp/cache/huggingface' >> ~/.bashrc
echo 'export TRANSFORMERS_CACHE=/workspace/pi-defense-exp/cache/huggingface' >> ~/.bashrc
echo 'export HF_DATASETS_CACHE=/workspace/pi-defense-exp/cache/huggingface/datasets' >> ~/.bashrc
echo 'export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True' >> ~/.bashrc
echo 'MODEL_ID=meta-llama/Meta-Llama-3.1-8B-Instruct' >> ~/.bashrc
```

In [7]:
PROJECT_DIR = Path("/workspace/pi-defense-exp")

CACHE_DIR = PROJECT_DIR / "cache" / "huggingface"
HF_DATASETS_DIR = CACHE_DIR / "datasets"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_DATASETS_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR)
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MODEL_ID"] = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [8]:
cache_env_vars = [
    "HF_HOME",
    "TRANSFORMERS_CACHE",
    "HF_DATASETS_CACHE",
    "PYTORCH_CUDA_ALLOC_CONF",
    "MODEL_ID",
]

env_rows = []

for var in cache_env_vars:
    env_rows.append({
        "variable": var,
        "value": os.environ.get(var, None)
    })

pd.DataFrame(env_rows)

,variable,value
0,HF_HOME,/workspace/pi-defense-exp/cache/huggingface
1,TRANSFORMERS_CACHE,/workspace/pi-defense-exp/cache/huggingface
2,HF_DATASETS_CACHE,/workspace/pi-defense-exp/cache/huggingface/da...
3,PYTORCH_CUDA_ALLOC_CONF,expandable_segments:True
4,MODEL_ID,meta-llama/Meta-Llama-3.1-8B-Instruct


## 7. Verificação da GPU

A GPU foi verificada usando `nvidia-smi` e PyTorch.

Comando utilizado no terminal:

```bash
nvidia-smi
```

In [9]:
try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        check=True,
    )
    print(result.stdout)
except Exception as e:
    print("Não foi possível executar nvidia-smi.")
    print(e)

Sat Jun 20 18:46:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000001:BE:00.0 Off |                  N/A |
|  0%   32C    P8             26W /  575W |       0MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
import torch

gpu_info = {
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

pd.DataFrame([gpu_info])

,torch_version,cuda_available,cuda_version,gpu_count,gpu_name
0,2.12.1+cu130,True,13.0,1,NVIDIA GeForce RTX 5090


## 8. Registro do ambiente

Para garantir reprodutibilidade, as versões das dependências, a versão do Python e as informações da GPU foram salvas em arquivos dentro de `results/environment`.

Comandos equivalentes utilizados:

```bash
pip freeze > results/environment/requirements-freeze.txt
python --version > results/environment/python-version.txt
nvidia-smi > results/environment/nvidia-smi.txt
```

In [11]:
ENVIRONMENT_DIR = PROJECT_DIR / "results" / "environment"
ENVIRONMENT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

requirements_path = ENVIRONMENT_DIR / f"requirements-freeze_{timestamp}.txt"
python_version_path = ENVIRONMENT_DIR / f"python-version_{timestamp}.txt"
nvidia_smi_path = ENVIRONMENT_DIR / f"nvidia-smi_{timestamp}.txt"
package_versions_path = ENVIRONMENT_DIR / f"package-versions_{timestamp}.csv"

# pip freeze
try:
    freeze_result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        check=True,
    )
    requirements_path.write_text(freeze_result.stdout, encoding="utf-8")
except Exception as e:
    print("Erro ao salvar pip freeze:", e)

# Python version
python_version_path.write_text(platform.python_version(), encoding="utf-8")

# nvidia-smi
try:
    smi_result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        check=True,
    )
    nvidia_smi_path.write_text(smi_result.stdout, encoding="utf-8")
except Exception as e:
    nvidia_smi_path.write_text(f"nvidia-smi unavailable: {e}", encoding="utf-8")

# package versions
packages_df.to_csv(package_versions_path, index=False)

print("Arquivos salvos:")
print(requirements_path)
print(python_version_path)
print(nvidia_smi_path)
print(package_versions_path)

Arquivos salvos:
/workspace/pi-defense-exp/results/environment/requirements-freeze_2026-06-20_18-46-54.txt
/workspace/pi-defense-exp/results/environment/python-version_2026-06-20_18-46-54.txt
/workspace/pi-defense-exp/results/environment/nvidia-smi_2026-06-20_18-46-54.txt
/workspace/pi-defense-exp/results/environment/package-versions_2026-06-20_18-46-54.csv


In [12]:
saved_environment_files = sorted(ENVIRONMENT_DIR.glob("*"))

pd.DataFrame([
    {
        "file": path.name,
        "size_kb": round(path.stat().st_size / 1024, 2),
        "path": str(path),
    }
    for path in saved_environment_files
])

,file,size_kb,path
0,nvidia-smi_2026-06-20_18-46-54.txt,1.65,/workspace/pi-defense-exp/results/environment/...
1,package-versions_2026-06-20_18-46-54.csv,0.29,/workspace/pi-defense-exp/results/environment/...
2,python-version_2026-06-20_18-46-54.txt,0.01,/workspace/pi-defense-exp/results/environment/...
3,requirements-freeze_2026-06-20_18-46-54.txt,3.16,/workspace/pi-defense-exp/results/environment/...
